 ### Import libraries

In [1]:
import pandas as pd
import matplotlib as plt
import seaborn as sns
import numpy as np

In [2]:
# import dataset
df = pd.read_csv('C:/Users/malla/Downloads/Dataset_ATS_v2 (1).csv')
df.head()

,gender,SeniorCitizen,Dependents,tenure,PhoneService,MultipleLines,InternetService,Contract,MonthlyCharges,Churn
0,Female,0,No,1,No,No,DSL,Month-to-month,25,Yes
1,Male,0,No,41,Yes,No,DSL,One year,25,No
2,Female,0,Yes,52,Yes,No,DSL,Month-to-month,19,No
3,Female,0,No,1,Yes,No,DSL,One year,76,Yes
4,Male,0,No,67,Yes,No,Fiber optic,Month-to-month,51,No


### Understanding the given data 

In [3]:
# finding the no of columns and rows

df.shape

(7043, 10)

### there are 10 colums and 7043 rows

In [4]:
# Overall information of dataset. info() shows the column types and non-null counts
df.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   gender           7043 non-null   object
 1   SeniorCitizen    7043 non-null   int64 
 2   Dependents       7043 non-null   object
 3   tenure           7043 non-null   int64 
 4   PhoneService     7043 non-null   object
 5   MultipleLines    7043 non-null   object
 6   InternetService  7043 non-null   object
 7   Contract         7043 non-null   object
 8   MonthlyCharges   7043 non-null   int64 
 9   Churn            7043 non-null   object
dtypes: int64(3), object(7)
memory usage: 550.4+ KB


In [5]:
# Finding numeric variable or columns
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.758768
std,0.368612,24.559481,30.091650
min,0.000000,0.000000,18.000000
25%,0.000000,9.000000,36.000000
50%,0.000000,29.000000,70.000000
75%,0.000000,55.000000,90.000000
max,1.000000,72.000000,119.000000


### Check for missing or incomplete values

In [6]:
# check null vaules in the dataset
df.isna().sum()

gender             0
SeniorCitizen      0
Dependents         0
tenure             0
PhoneService       0
MultipleLines      0
InternetService    0
Contract           0
MonthlyCharges     0
Churn              0
dtype: int64

### Feature Engineering

#### Select relevant features
Our goal is to predict whether a telephone customer will churn. We use the nine available predictors because each could potentially relate to churn:


In [7]:
# Grouping categorical value in dataset
categorical_columns = df.select_dtypes(include=['object'], exclude=['int64'])
categorical_columns

,gender,Dependents,PhoneService,MultipleLines,InternetService,Contract,Churn
0,Female,No,No,No,DSL,Month-to-month,Yes
1,Male,No,Yes,No,DSL,One year,No
2,Female,Yes,Yes,No,DSL,Month-to-month,No
3,Female,No,Yes,No,DSL,One year,Yes
4,Male,No,Yes,No,Fiber optic,Month-to-month,No
...,...,...,...,...,...,...,...
7038,Male,No,Yes,Yes,DSL,One year,Yes
7039,Female,Yes,Yes,Yes,Fiber optic,One year,No
7040,Male,Yes,Yes,No,DSL,Month-to-month,No
7041,Male,No,Yes,Yes,Fiber optic,Month-to-month,Yes


In [8]:
# finding numeric value in dataset
numerical_columns =  df.select_dtypes(include=['int64'], exclude=['object'])
numerical_columns

,SeniorCitizen,tenure,MonthlyCharges
0,0,1,25
1,0,41,25
2,0,52,19
3,0,1,76
4,0,67,51
...,...,...,...
7038,0,1,95
7039,0,23,91
7040,0,12,21
7041,1,12,99


In [9]:
df_cleaned = df.copy()


### Our goal is to predict whether a telephone customer will churn. We use the nine available predictors because each could potentially relate to churn:
We exclude Churn from the input features because it is the outcome we want to predict. We can later examine feature importance to decide which predictors are genuinely useful.

In [10]:
features = [
    "gender", "SeniorCitizen", "Dependents", "tenure",
    "PhoneService", "MultipleLines", "InternetService",
    "Contract", "MonthlyCharges"]

In [11]:
print("Number of predictor columns:", len(features))

Number of predictor columns: 9


### creating new features:

We create two extra variables without using the Churn result:

estimated_total_charges = tenure × MonthlyCharges. This approximates a customer's overall billed amount. It is not an actual historical total because monthly charges may change over time.

tenure_group puts customers into four tenure ranges (0–12, 13–24, 25–48, and 49+ months). This can help a model capture non-linear relationships, such as a different churn pattern for newer customers.
These calculations are based on individual customer values or fixed cut-offs, not statistics learnt from the test dataset.

In [12]:
df_cleaned["estimated_total_charges"] = (
    df_cleaned["tenure"] * df_cleaned["MonthlyCharges"]
)

# Group customers by how long they have been with the company.
df_cleaned["tenure_group"] = pd.cut(
    df_cleaned["tenure"],
    bins=[-1, 12, 24, 48, np.inf],
    labels=["0-12 months", "13-24 months", "25-48 months", "49+ months"]
)


In [13]:
features =features + [
    "estimated_total_charges", "tenure_group"
]

print("Number of predictors after feature engineering:", len(features))
display(df_cleaned.head())

Number of predictors after feature engineering: 11


,gender,SeniorCitizen,Dependents,tenure,PhoneService,MultipleLines,InternetService,Contract,MonthlyCharges,Churn,estimated_total_charges,tenure_group
0,Female,0,No,1,No,No,DSL,Month-to-month,25,Yes,25,0-12 months
1,Male,0,No,41,Yes,No,DSL,One year,25,No,1025,25-48 months
2,Female,0,Yes,52,Yes,No,DSL,Month-to-month,19,No,988,49+ months
3,Female,0,No,1,Yes,No,DSL,One year,76,Yes,76,0-12 months
4,Male,0,No,67,Yes,No,Fiber optic,Month-to-month,51,No,3417,49+ months


In [14]:
display(df_cleaned.head())

,gender,SeniorCitizen,Dependents,tenure,PhoneService,MultipleLines,InternetService,Contract,MonthlyCharges,Churn,estimated_total_charges,tenure_group
0,Female,0,No,1,No,No,DSL,Month-to-month,25,Yes,25,0-12 months
1,Male,0,No,41,Yes,No,DSL,One year,25,No,1025,25-48 months
2,Female,0,Yes,52,Yes,No,DSL,Month-to-month,19,No,988,49+ months
3,Female,0,No,1,Yes,No,DSL,One year,76,Yes,76,0-12 months
4,Male,0,No,67,Yes,No,Fiber optic,Month-to-month,51,No,3417,49+ months


### Define X and y
y represents the dependent variable (output) that you want to predict or analyze against your features i.e X
 to
We encode only the target at this stage: No = 0 and Yes = 1 to avoid data leakage.

In [15]:
# X represent the feature varible
X = df_cleaned[features].copy()

# y represent target variable
y = df_cleaned["Churn"].map({"No": 0, "Yes": 1}).astype(int)

In [16]:
X

,gender,SeniorCitizen,Dependents,tenure,PhoneService,MultipleLines,InternetService,Contract,MonthlyCharges,estimated_total_charges,tenure_group
0,Female,0,No,1,No,No,DSL,Month-to-month,25,25,0-12 months
1,Male,0,No,41,Yes,No,DSL,One year,25,1025,25-48 months
2,Female,0,Yes,52,Yes,No,DSL,Month-to-month,19,988,49+ months
3,Female,0,No,1,Yes,No,DSL,One year,76,76,0-12 months
4,Male,0,No,67,Yes,No,Fiber optic,Month-to-month,51,3417,49+ months
...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,No,1,Yes,Yes,DSL,One year,95,95,0-12 months
7039,Female,0,Yes,23,Yes,Yes,Fiber optic,One year,91,2093,13-24 months
7040,Male,0,Yes,12,Yes,No,DSL,Month-to-month,21,252,0-12 months
7041,Male,1,No,12,Yes,Yes,Fiber optic,Month-to-month,99,1188,0-12 months


### Splitting dataset into Training set and Test sets
We splitting our dataset X and y into two groups:

X_train will contain 80% data of X dataset which will be used for training X_test will contain 20% data of X dataset which will be used for training y-train will contain 80% data of y dataset which will be used for training y-test will contain 20% data of y dataset which will be used for training

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


In [18]:
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 5634
Testing rows: 1409


### Feature encoding

Different columns need different preprocessing:

- Numerical predictors: We use StandardScaler so numerical variables have a mean of roughly 0 and standard deviation 1 in the training set.
- 
  Categorical predictors: We apply OneHotEncoder to make 0/1 indicator columns. 
- SeniorCitizen: it is already a 0/1 

A ColumnTransformer applies the correct steps to each feature group. The same fitted transformations will later be applied to test customers.

In [19]:
# The columns are grouped according to the processing they need.
numerical_features = [
    "tenure", "MonthlyCharges", "estimated_total_charges"
]

categorical_features = [
    "gender", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "Contract", "tenure_group"
]

binary_features = ["SeniorCitizen"]

### Enconding each features  types

Numerical columns need scaling because larger numbers like  tenure, monthlyCharges disproportionately dominate smaller numbers like tenure or binary flags).

Categorical columns need encoding because machine learning algorithms require numbers instead of text strings like "Fiber optic" or "Month-to-month").


### Fit on training data and transform the test data

We use fit_transform() once on X_train. It learns training medians, category mappings, and scaling values.

We use only transform() on X_test. Doing so would avoid inconsistent scaling and data leakage.

In [24]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import pandas as pd

# Create scaler
scaler = StandardScaler()

# Scale numerical features
X_train_num = scaler.fit_transform(X_train[numerical_features])
X_test_num = scaler.transform(X_test[numerical_features])

# Convert into DataFrames
X_train_num = pd.DataFrame(X_train_num, columns=numerical_features, index=X_train.index)
X_test_num = pd.DataFrame(X_test_num, columns=numerical_features, index=X_test.index)

## Categorical encoding

In [25]:
# Create encoder
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# Encode categorical features
X_train_cat = encoder.fit_transform(X_train[categorical_features])
X_test_cat = encoder.transform(X_test[categorical_features])

# Convert into DataFrames
encoded_columns = encoder.get_feature_names_out(categorical_features)

X_train_cat = pd.DataFrame(X_train_cat, columns=encoded_columns, index=X_train.index)
X_test_cat = pd.DataFrame(X_test_cat, columns=encoded_columns, index=X_test.index)

In [29]:
X_train_binary = X_train[binary_features].copy()
X_test_binary = X_test[binary_features].copy()

### Combining processed data

In [30]:
# Combine processed training data
X_train_processed = pd.concat([
    X_train_num,
    X_train_cat,
    X_train_binary
], axis=1)

# Combine processed testing data
X_test_processed = pd.concat([
    X_test_num,
    X_test_cat,
    X_test_binary
], axis=1)

print("Training data:", X_train_processed.shape)
print("Testing data:", X_test_processed.shape)

# Display first five rows
X_train_processed.head()

Training data: (5634, 21)
Testing data: (1409, 21)


,tenure,MonthlyCharges,estimated_total_charges,gender_Female,gender_Male,Dependents_No,Dependents_Yes,PhoneService_No,PhoneService_Yes,MultipleLines_No,...,InternetService_DSL,InternetService_Fiber optic,Contract_Month-to-month,Contract_One year,Contract_Two year,tenure_group_0-12 months,tenure_group_13-24 months,tenure_group_25-48 months,tenure_group_49+ months,SeniorCitizen
3757,-1.276682,-0.149881,-0.976299,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0
3165,-0.096089,-0.315125,-0.273851,1.0,0.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0
4912,0.555273,1.172071,1.029433,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1
3877,0.433143,0.048412,0.250997,0.0,1.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0
3818,-1.276682,-1.471834,-0.993971,0.0,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0


## Check the final prepared datasets

At this point, I check that both datasets have the same columns, contain only numerical values and have no missing values. These checks help identify errors before training a model.

In [36]:
# Check the columns match
print("Same columns:", X_train_processed.columns.equals(X_test_processed.columns))

# Check if missing values remain
print("Missing training values:", X_train_processed.isna().sum().sum())
print("Missing testing values:", X_test_processed.isna().sum().sum())

Same columns: True
Missing training values: 0
Missing testing values: 0


In [34]:
 X_train_processed

,tenure,MonthlyCharges,estimated_total_charges,gender_Female,gender_Male,Dependents_No,Dependents_Yes,PhoneService_No,PhoneService_Yes,MultipleLines_No,...,InternetService_DSL,InternetService_Fiber optic,Contract_Month-to-month,Contract_One year,Contract_Two year,tenure_group_0-12 months,tenure_group_13-24 months,tenure_group_25-48 months,tenure_group_49+ months,SeniorCitizen
3757,-1.276682,-0.149881,-0.976299,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0
3165,-0.096089,-0.315125,-0.273851,1.0,0.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0
4912,0.555273,1.172071,1.029433,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1
3877,0.433143,0.048412,0.250997,0.0,1.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0
3818,-1.276682,-1.471834,-0.993971,0.0,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6335,-0.951001,-0.315125,-0.784120,1.0,0.0,0.0,1.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0
6153,-1.235972,-0.348174,-0.955093,0.0,1.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0
4508,-0.828871,-0.249028,-0.700622,0.0,1.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0
2727,0.107462,-1.471834,-0.693553,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0


## Exporting the traning and test datasets

In [ ]:
X_train_processed.to_csv('C:/Users/malla/X_train_standardised.csv', index=False)
X_test_processed.to_csv('C:/Users/malla/ X_test_standardised.csv', index=False)
y_train.to_csv('C:/Users/malla/y_train.csv', index=False)
y_test.to_csv('C:/Users/malla/y_test.csv', index=False)

### Purpose of Saving the instance of scaler
a. When you .fit_transform() your training data, the scaler learns specific parameters—namely, the mean and standard deviation of each column in your training set.

b.When new, unseen customer data comes in later (or when you evaluate your model on your test set), that new data must be scaled using the exact same mean and standard deviation learned from the training data.

c. Saving the scaler and encoder ensures you can load it later using joblib.load('scaler.pkl') to transform new inputs identically.


In [46]:
import joblib

joblib.dump(scaler,'C:/Users/malla/standard_scaler.pkl')
joblib.dump(encoder,'C:/Users/malla/onehot_encoder.pkl')


['C:/Users/malla/onehot_encoder.pkl']

'C:\\Users\\malla'